In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
SupportsFloat = float
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

NANX_RESULTS_DIR = Path("/tmp/plaintext_sims_results")
NANX_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

_NANX_RESULTS_ROWS = {
    "total_calendar_time": [114.9, 103.0],
    "num_rework_cycles": [0.6, 1.2],
    "prop_defects_caught_early": [0.82, 0.73],
    "num_late_defects": [0.3, 0.9],
    "handover_delay": [0.5, 2.2],
    "condition": ["plaintext", "mixed"],
    "seed": [279223610, 423198623],
}
_NANX_BOOT_ROWS = [
    {"metric": "total_calendar_time", "delta_mean": 11.868794921705613, "ci_lower": 11.371132533828291, "ci_upper": 12.387187407199493},
    {"metric": "num_rework_cycles", "delta_mean": -0.5644594, "ci_lower": -0.5884025, "ci_upper": -0.5410000000000001},
]

# --- run_concat_write ---
FIX_RUN_CONCAT_WRITE_RESULTS_DIR = NANX_RESULTS_DIR
FIX_RUN_CONCAT_WRITE_SIM_MIXED_BEFORE = pd.DataFrame(_NANX_RESULTS_ROWS).query("condition == 'mixed'").reset_index(drop=True)
FIX_RUN_CONCAT_WRITE_SIM_PLAINTEXT_BEFORE = pd.DataFrame(_NANX_RESULTS_ROWS).query("condition == 'plaintext'").reset_index(drop=True)
FIX_RUN_CONCAT_WRITE_SIM_MIXED_GEN = pl.from_pandas(FIX_RUN_CONCAT_WRITE_SIM_MIXED_BEFORE)
FIX_RUN_CONCAT_WRITE_SIM_PLAINTEXT_GEN = pl.from_pandas(FIX_RUN_CONCAT_WRITE_SIM_PLAINTEXT_BEFORE)

# --- run_dataframe_write ---
FIX_RUN_DATAFRAME_WRITE_RESULTS_DIR = NANX_RESULTS_DIR
FIX_RUN_DATAFRAME_WRITE_SIM_BOOT = _NANX_BOOT_ROWS

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_run_concat_write(results_dir, sim_mixed, sim_plaintext):
    sim_results = pd.concat([sim_plaintext, sim_mixed], ignore_index=True)
    sim_results.to_csv(results_dir / "simpy_results.csv", index=False)
    return sim_results

def before_run_dataframe_write(results_dir, sim_boot):
    pd.DataFrame(sim_boot).to_csv(
        results_dir / "simpy_bootstrap_effects.csv", index=False
    )
    return None

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_run_concat_write(results_dir, sim_mixed, sim_plaintext):
    sim_results = pl.concat([sim_plaintext, sim_mixed], how="vertical")
    sim_results.write_csv(results_dir / "simpy_results.csv")
    return sim_results

def gen_run_dataframe_write(results_dir, sim_boot):
    pl.DataFrame(sim_boot).write_csv(
        results_dir / "simpy_bootstrap_effects.csv"
    )
    return None

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: run_dataframe_write ===

# L1 smoke – generated
try:
    _r = gen_run_dataframe_write(FIX_RUN_DATAFRAME_WRITE_RESULTS_DIR, FIX_RUN_DATAFRAME_WRITE_SIM_BOOT)
    print("✅ L1 smoke gen_run_dataframe_write: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_run_dataframe_write: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_run_dataframe_write(FIX_RUN_DATAFRAME_WRITE_RESULTS_DIR, FIX_RUN_DATAFRAME_WRITE_SIM_BOOT)
    print("✅ L1 smoke before_run_dataframe_write: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_run_dataframe_write: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence – compare file contents written by both paths.
try:
    _before_dir = Path("/tmp/plaintext_sims_before_l2")
    _gen_dir = Path("/tmp/plaintext_sims_gen_l2")
    _before_dir.mkdir(parents=True, exist_ok=True)
    _gen_dir.mkdir(parents=True, exist_ok=True)
    before_run_dataframe_write(_before_dir, FIX_RUN_DATAFRAME_WRITE_SIM_BOOT)
    gen_run_dataframe_write(_gen_dir, FIX_RUN_DATAFRAME_WRITE_SIM_BOOT)
    _before_csv = pd.read_csv(_before_dir / "simpy_bootstrap_effects.csv")
    _gen_csv = pl.read_csv(_gen_dir / "simpy_bootstrap_effects.csv")
    compare(_before_csv, _gen_csv, "run_dataframe_write")
except Exception as _e:
    print(f"❌ L2 equivalence run_dataframe_write: setup error — {type(_e).__name__}: {_e}")

# L3 edge - compare header-only CSV output on both sides.
import tempfile
try:
    with tempfile.TemporaryDirectory() as _bd, tempfile.TemporaryDirectory() as _gd:
        _bdir, _gdir = Path(_bd), Path(_gd)
        before_run_dataframe_write(_bdir, FIX_RUN_DATAFRAME_WRITE_SIM_BOOT[:0])
        gen_run_dataframe_write(_gdir, FIX_RUN_DATAFRAME_WRITE_SIM_BOOT[:0])
        _bp, _gp = _bdir / "simpy_bootstrap_effects.csv", _gdir / "simpy_bootstrap_effects.csv"
        _bt, _gt = _bp.read_text(), _gp.read_text()
        if _bt == _gt:
            print("✅ L3 edge run_dataframe_write empty CSV: MATCH")
        else:
            print(f"❌ L3 edge run_dataframe_write empty CSV: MISMATCH — before={_bt!r}, gen={_gt!r}")
except Exception as _e:
    print(f"❌ L3 edge run_dataframe_write: {type(_e).__name__}: {_e}")
